In [1]:
# obtain a list of files in the input directory
import os

files_in_input_dir=os.listdir('data/input/')
files_in_input_dir

['file2.txt', 'file3.txt', 'file1.txt', 'file4.txt']

In [2]:
# count the frequency of the words in the files in the input directory
counter={}
for filename in files_in_input_dir:
    with open('data/input/'+filename) as f:
        for l in f:
            for w in l.split( ):
                w = w.lower().strip(",.!?")
                counter[w] = counter.get(w, 0) + 1

In [3]:
# create the directory output/ if it doesn't exist
if not os.path.exists('data/output'):
    os.makedirs('data/output')

# save the results using tsv format
with open("data/output/results.tsv", "w", encoding="utf-8") as f:
        for key, value in counter.items():
            # write the key and value to the file
            f.write(f"{key}\t{value}\n")



## Pruebas Unitarias

A continuación se prueban individualmente cada una de las funciones del módulo `homework/src/_internals/`.

In [ ]:
import unittest
import os
import tempfile

from homework.src._internals.preprocess_lines import preprocess_lines
from homework.src._internals.split_into_words import split_into_words
from homework.src._internals.count_words import count_words
from homework.src._internals.write_word_counts import write_word_counts
from homework.src._internals.read_all_lines import read_all_lines


class TestPreprocessLines(unittest.TestCase):

    def test_lowercase_conversion(self):
        result = preprocess_lines(["Hello World", "FOO BAR"])
        self.assertEqual(result, ["hello world", "foo bar"])

    def test_strip_whitespace(self):
        result = preprocess_lines(["  hello  ", "  world  "])
        self.assertEqual(result, ["hello", "world"])

    def test_empty_list(self):
        result = preprocess_lines([])
        self.assertEqual(result, [])

    def test_already_lowercase(self):
        result = preprocess_lines(["python", "unittest"])
        self.assertEqual(result, ["python", "unittest"])


class TestSplitIntoWords(unittest.TestCase):

    def test_basic_split(self):
        result = split_into_words(["hello world"])
        self.assertEqual(result, ["hello", "world"])

    def test_removes_punctuation(self):
        result = split_into_words(["hello, world!"])
        self.assertEqual(result, ["hello", "world"])

    def test_multiple_lines(self):
        result = split_into_words(["foo bar", "baz qux"])
        self.assertEqual(result, ["foo", "bar", "baz", "qux"])

    def test_empty_list(self):
        result = split_into_words([])
        self.assertEqual(result, [])


class TestCountWords(unittest.TestCase):

    def test_single_word(self):
        result = count_words(["hello"])
        self.assertEqual(result, {"hello": 1})

    def test_repeated_words(self):
        result = count_words(["hello", "world", "hello"])
        self.assertEqual(result, {"hello": 2, "world": 1})

    def test_empty_list(self):
        result = count_words([])
        self.assertEqual(result, {})

    def test_multiple_words(self):
        words = ["a", "b", "a", "c", "b", "a"]
        result = count_words(words)
        self.assertEqual(result, {"a": 3, "b": 2, "c": 1})


class TestWriteWordCounts(unittest.TestCase):

    def test_creates_output_file(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            write_word_counts(tmpdir, {"hello": 2, "world": 1})
            output_file = os.path.join(tmpdir, "wordcount.tsv")
            self.assertTrue(os.path.exists(output_file))

    def test_correct_file_content(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            write_word_counts(tmpdir, {"hello": 3})
            output_file = os.path.join(tmpdir, "wordcount.tsv")
            with open(output_file, "r", encoding="utf-8") as f:
                content = f.read()
            self.assertIn("hello\t3", content)

    def test_creates_output_dir_if_missing(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            new_dir = os.path.join(tmpdir, "subdir")
            write_word_counts(new_dir, {"test": 1})
            self.assertTrue(os.path.exists(new_dir))


class TestReadAllLines(unittest.TestCase):

    def test_reads_files_from_input_folder(self):
        lines = read_all_lines("data/input")
        self.assertIsInstance(lines, list)
        self.assertGreater(len(lines), 0)

    def test_lines_are_strings(self):
        lines = read_all_lines("data/input")
        for line in lines:
            self.assertIsInstance(line, str)

    def test_reads_from_temp_folder(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            temp_file = os.path.join(tmpdir, "test.txt")
            with open(temp_file, "w", encoding="utf-8") as f:
                f.write("line one\nline two\n")
            lines = read_all_lines(tmpdir)
            self.assertEqual(lines, ["line one\n", "line two\n"])


unittest.main(argv=[''], exit=False, verbosity=2)